# Egyptora — Free Data Collection & Scraping Notebook

هذه الـNotebook هي المرجع القابل لإعادة التشغيل لجمع البيانات المجانية المستخدمة في مشروع **Egyptora**. تجمع البيانات من المصادر الرسمية أو المفتوحة التالية:

- وزارة السياحة والآثار المصرية (MOTA): المنشآت الفندقية الثابتة، المواقع الأثرية، والمتاحف.
- World Bank API: السائحون الدوليون وإيرادات السياحة المتاحة.
- المصادر الحكومية المصرية المنشورة: استكمال أعداد السائحين 2020–2025 مع توثيق درجة الدقة.
- NASA POWER API: مناخ شهري لنقاط الوجهات.
- Wikimedia Pageviews API: الاهتمام الرقمي بصفحات المعالم.
- geoBoundaries API: حدود المحافظات ADM1.
- UNESCO World Heritage XML: مواقع التراث العالمي في مصر.
- OpenStreetMap Overpass API: مرجع مفتوح للمعالم والإقامة (اختياري بسبب حدود الخادم العام).

## قواعد مهمة

1. لا تكتب الـNotebook فوق فولدر `Data` النهائي؛ كل ناتج جديد يُحفظ في `Scraping/output`.
2. البيانات الخام تحفظ في `output/raw`، والجداول المنظمة في `output/normalized`، ومراجعات الاستبعاد في `output/exclusions`.
3. Pageviews تعني اهتمامًا رقميًا وليست زيارات سياحية.
4. NASA POWER بيانات مناخية نموذجية من MERRA-2 وليست قياسات محطة محلية.
5. OSM مرجع مفتوح وليس سجلًا حكوميًا شاملًا.
6. قيم 2020–2022 و2024–2025 منشورة بصورة مقربة؛ 2023 منشور كقيمة محددة 14.906 مليون.
7. لا يتم اختلاق إيرادات 2021–2025؛ تظل فارغة إن لم توجد سلسلة سنوية تقويمية قابلة للمقارنة.

In [ ]:
from pathlib import Path
from datetime import date, datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.request import Request, urlopen
from urllib.parse import urlencode, quote, urlsplit, urlunsplit
from urllib.error import HTTPError, URLError
from xml.etree import ElementTree as ET
from collections import Counter
import csv
import hashlib
import json
import math
import re
import time

# Paths: keep new downloads separate from the approved Data folder.
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "scraping":
    candidate = NOTEBOOK_DIR / "New folder (3)" / "Scraping"
    if candidate.exists():
        NOTEBOOK_DIR = candidate.resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / "Data"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
RAW_DIR = OUTPUT_DIR / "raw"
NORMALIZED_DIR = OUTPUT_DIR / "normalized"
EXCLUSIONS_DIR = OUTPUT_DIR / "exclusions"
for folder in (OUTPUT_DIR, RAW_DIR, NORMALIZED_DIR, EXCLUSIONS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Expected approved input folder: {DATA_DIR}")

# Main switches. OSM is complete but disabled by default to respect public Overpass limits.
RUN_WORLD_BANK = True
RUN_OFFICIAL_TOURISM_PAGES = True
RUN_MOTA = True
RUN_GEOBOUNDARIES = True
RUN_NASA_POWER = True
RUN_WIKIMEDIA = True
RUN_UNESCO = True
RUN_OSM = False

USER_AGENT = "Egyptora-graduation-project-data-audit/2.0 (academic data collection)"
NOW_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
START_YEAR = 2019
END_YEAR = 2025

first_day_this_month = date.today().replace(day=1)
last_complete_day = first_day_this_month - timedelta(days=1)
WIKIMEDIA_START = "2019010100"
WIKIMEDIA_END = f"{last_complete_day:%Y%m%d}00"
COMMON_WINDOW_START = "2024-07-01"
COMMON_WINDOW_END = f"{last_complete_day:%Y-%m}-01"

print(f"Data input: {DATA_DIR}")
print(f"New output: {OUTPUT_DIR}")
print(f"Run timestamp: {NOW_UTC}")
print(f"Last complete Wikimedia month: {COMMON_WINDOW_END}")

## أدوات الاتصال والحفظ

كل الاتصالات تستخدم مهلة زمنية، ومحاولات متكررة بتأخير تدريجي، وUser-Agent واضح. يتم حفظ الخام قبل تحويله إلى CSV، ويُنشأ في النهاية manifest ببصمة SHA-256 لكل ملف.

In [ ]:
COLLECTION_EVENTS = []

def http_bytes(url, method="GET", json_body=None, headers=None, attempts=4, timeout=120):
    # Convert Arabic/non-ASCII paths to a valid HTTP URL while preserving percent escapes.
    parts = urlsplit(url)
    url = urlunsplit((
        parts.scheme,
        parts.netloc.encode("idna").decode("ascii"),
        quote(parts.path, safe="/%:@"),
        quote(parts.query, safe="=&%:@,+?"),
        parts.fragment,
    ))
    request_headers = {"User-Agent": USER_AGENT, "Accept": "*/*"}
    if headers:
        request_headers.update(headers)
    data = None
    if json_body is not None:
        data = json.dumps(json_body, ensure_ascii=False).encode("utf-8")
        request_headers.setdefault("Content-Type", "application/json")
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            req = Request(url, data=data, method=method, headers=request_headers)
            with urlopen(req, timeout=timeout) as response:
                return response.read(), response.geturl(), dict(response.headers)
        except Exception as exc:
            last_error = exc
            if attempt == attempts:
                break
            if isinstance(exc, HTTPError) and exc.code == 429:
                retry_after = exc.headers.get("Retry-After", "") if exc.headers else ""
                try:
                    delay = max(5, int(retry_after))
                except (TypeError, ValueError):
                    delay = min(90, 15 * attempt)
            else:
                delay = min(30, 2 * attempt)
            time.sleep(delay)
    raise RuntimeError(f"Request failed after {attempts} attempts: {url} :: {last_error}")

def http_json(url, **kwargs):
    payload, final_url, headers = http_bytes(url, **kwargs)
    return json.loads(payload.decode("utf-8-sig")), final_url, headers

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def write_text(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(value, encoding="utf-8")
    return path

def write_csv(path, rows, fieldnames=None):
    path = Path(path)
    rows = list(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    fieldnames.append(key)
                    seen.add(key)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    return path

def read_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()

def stable_id(prefix, text):
    digest = hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16].upper()
    return f"{prefix}{digest}"

def safe_filename(text):
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")
    return value[:120] or "record"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def event(source_id, status, rows=0, note=""):
    COLLECTION_EVENTS.append({
        "source_id": source_id,
        "status": status,
        "rows": rows,
        "note": note,
        "event_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    })

def optional_html_snapshot(source_id, url):
    try:
        payload, final_url, headers = http_bytes(url, timeout=90)
        path = RAW_DIR / "official_tourism_pages" / f"{source_id}.html"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)
        event(source_id, "SNAPSHOT_DOWNLOADED", 1, final_url)
        return "DOWNLOADED", str(path.relative_to(OUTPUT_DIR))
    except Exception as exc:
        event(source_id, "SNAPSHOT_UNAVAILABLE_VALUE_RETAINED", 0, str(exc))
        return "UNAVAILABLE", str(exc)

print("Helpers ready — standard library only.")

## سجل المصادر

هذا السجل يوضح الجهة، وطريقة الجمع، والتكلفة، وحدود الاستخدام التحليلي. وجود المصدر هنا لا يعني أن كل مقياس يصلح كمقياس طلب سياحي.

In [ ]:
SOURCE_REGISTER = [
    {"source_id":"SRC_MOTA_DIRECTORIES","provider":"Egypt Ministry of Tourism and Antiquities","dataset":"Fixed accommodation; archaeology; museums","method":"POST public directory endpoint","cost":"FREE_PUBLIC","license_or_terms":"Public official directory","source_url":"https://mota.gov.eg/GenericList/GetAll","analysis_role":"Official inventory and identity; not occupancy or demand."},
    {"source_id":"SRC_WB","provider":"World Bank WDI","dataset":"International arrivals and tourism receipts","method":"REST API JSON","cost":"FREE_API","license_or_terms":"CC BY 4.0","source_url":"https://api.worldbank.org/v2/country/EGY","analysis_role":"National annual trend where values are available."},
    {"source_id":"SRC_EGY_OFFICIAL_ANNUAL","provider":"Egyptian government sources","dataset":"International arrivals 2020-2025","method":"Official page snapshots + normalized published values","cost":"FREE_PUBLIC","license_or_terms":"Public government pages","source_url":"Multiple URLs stored per row","analysis_role":"Calendar-year arrivals; rounded rows are explicitly flagged."},
    {"source_id":"SRC_NASA_POWER","provider":"NASA Langley POWER","dataset":"Monthly destination climate","method":"REST API JSON","cost":"FREE_API","license_or_terms":"NASA open data terms","source_url":"https://power.larc.nasa.gov/api/temporal/monthly/point","analysis_role":"Modelled climate suitability; not station observation."},
    {"source_id":"SRC_WIKIMEDIA","provider":"Wikimedia Foundation","dataset":"Monthly pageviews","method":"REST API JSON","cost":"FREE_API","license_or_terms":"Wikimedia API terms","source_url":"https://wikimedia.org/api/rest_v1/","analysis_role":"Digital interest only; not tourist visits."},
    {"source_id":"SRC_GEOBOUNDARIES","provider":"geoBoundaries","dataset":"Egypt ADM1 boundaries","method":"REST metadata + GeoJSON","cost":"FREE_OPEN_DATA","license_or_terms":"CC BY 4.0","source_url":"https://www.geoboundaries.org/api/current/gbOpen/EGY/ADM1/","analysis_role":"Governorate spatial reference."},
    {"source_id":"SRC_UNESCO","provider":"UNESCO World Heritage Centre","dataset":"World Heritage sites in Egypt","method":"Official XML export","cost":"FREE_PUBLIC","license_or_terms":"UNESCO site terms","source_url":"https://whc.unesco.org/en/list/xml/","analysis_role":"Official heritage reference."},
    {"source_id":"SRC_OSM","provider":"OpenStreetMap contributors","dataset":"Attraction and accommodation candidates","method":"Overpass API","cost":"FREE_OPEN_DATA","license_or_terms":"ODbL","source_url":"https://overpass-api.de/api/interpreter","analysis_role":"Reference/candidate generation only; not an official census."},
]
write_csv(OUTPUT_DIR / "source_register.csv", SOURCE_REGISTER)
print(f"Sources documented: {len(SOURCE_REGISTER)}")

## 1) World Bank + الاستكمال الحكومي 2020–2025

يتم تنزيل مؤشري WDI:

- `ST.INT.ARVL`: International tourism, number of arrivals.
- `ST.INT.RCPT.CD`: International tourism receipts (current US$).

ثم تُضاف أعداد السائحين 2020–2025 من صفحات حكومية محددة. لا يتم مزج إيرادات السنة المالية مع السنة التقويمية، لذلك لا نملأ إيرادات 2021–2025 دون مصدر قابل للمقارنة.

In [ ]:
WB_INDICATORS = {
    "ST.INT.ARVL": ("International tourism, number of arrivals", "persons"),
    "ST.INT.RCPT.CD": ("International tourism, receipts (current US$)", "current_USD"),
}

def collect_world_bank():
    rows = []
    for indicator_code, (indicator_name, unit) in WB_INDICATORS.items():
        url = f"https://api.worldbank.org/v2/country/EGY/indicator/{indicator_code}?format=json&per_page=100"
        payload, final_url, _ = http_json(url, timeout=90)
        write_json(RAW_DIR / "world_bank" / f"{indicator_code}.json", payload)
        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f"Unexpected World Bank response: {indicator_code}")
        for item in payload[1] or []:
            value = item.get("value")
            if value is None:
                continue
            rows.append({
                "country_iso3":"EGY", "year":int(item["date"]),
                "indicator_code":indicator_code, "indicator_name":indicator_name,
                "value":value, "unit":unit, "source_name":"World Bank WDI",
                "source_url":url, "license_terms":"CC BY 4.0",
                "source_method":"AUTOMATED_OFFICIAL_API", "precision_flag":"AS_REPORTED",
                "downloaded_utc":NOW_UTC, "qa_status":"READY_OFFICIAL_AGGREGATE",
            })
    rows.sort(key=lambda r: (r["indicator_code"], r["year"]))
    write_csv(NORMALIZED_DIR / "fact_world_bank_tourism_annual.csv", rows)
    event("SRC_WB", "SUCCESS", len(rows), "Two WDI indicators; null API years excluded.")
    return rows

wb_rows = collect_world_bank() if RUN_WORLD_BANK else []
print(f"World Bank non-null rows: {len(wb_rows)}")

In [ ]:
OFFICIAL_ARRIVALS = [
    {"year":2020,"value":3700000,"precision_flag":"APPROXIMATE_PUBLISHED_3_7M","source_id":"SRC_MPED_2020_2021","source_name":"Egypt Ministry of Planning and Economic Development","source_url":"https://mped.gov.eg/singlenews?id=3612","evidence_note":"Published as 3.7 million tourists in 2020."},
    {"year":2021,"value":8000000,"precision_flag":"APPROXIMATE_PUBLISHED_8_0M","source_id":"SRC_MPED_2020_2021","source_name":"Egypt Ministry of Planning and Economic Development","source_url":"https://mped.gov.eg/singlenews?id=3612","evidence_note":"Published as 8 million tourists in 2021."},
    {"year":2022,"value":11700000,"precision_flag":"APPROXIMATE_PUBLISHED_11_7M","source_id":"SRC_CAPMAS_SIS_2022","source_name":"CAPMAS via Egypt State Information Service","source_url":"https://sis.gov.eg/es/centro-de-informacion/noticias-exclusivas/117-millones-de-turistas-visitaron-egipto-en-2022-y-los-ingresos-registraron-10700-millones-de-d%C3%B3lares/","evidence_note":"CAPMAS figure republished by SIS: 11.7 million in 2022."},
    {"year":2023,"value":14906000,"precision_flag":"EXACT_PUBLISHED_14_906M","source_id":"SRC_MOTA_2023","source_name":"Egypt Ministry of Tourism and Antiquities","source_url":"https://egymonuments.gov.eg/en/events/tourism-in-egypt-2023","evidence_note":"Published as 14.906 million tourists in 2023."},
    {"year":2024,"value":15800000,"precision_flag":"APPROXIMATE_PUBLISHED_15_8M","source_id":"SRC_MOTA_2024","source_name":"Egypt Ministry of Tourism and Antiquities","source_url":"https://mota.gov.eg/ar/الأخبار-2-1-1/وزير-السياحة-والآثار-يختتم-زيارته-الرسمية-إلى-صربيا-بلقاءات-إعلامية-موسعة-لتسليط-الضوء-على-آفاق-التعاون-السياحي-والثقافي-بين-البلدين/","evidence_note":"Published as about 15.8 million tourists in 2024."},
    {"year":2025,"value":19000000,"precision_flag":"APPROXIMATE_PUBLISHED_19_0M","source_id":"SRC_MOTA_2025","source_name":"Egypt Ministry of Tourism and Antiquities","source_url":"https://mota.gov.eg/ar/الأخبار-2-1-1/قفزة-جديدة-للسياحة-المصرية-في-2025-مصر-تستقبل-19-مليون-سائح-بمعدل-نمو-21-مقارنة-بعام-2024/","evidence_note":"Published as 19 million tourists in 2025."},
]

snapshot_rows = []
if RUN_OFFICIAL_TOURISM_PAGES:
    for row in OFFICIAL_ARRIVALS:
        status, detail = optional_html_snapshot(row["source_id"], row["source_url"])
        snapshot_rows.append({"source_id":row["source_id"],"year":row["year"],"snapshot_status":status,"detail":detail,"source_url":row["source_url"]})
write_csv(OUTPUT_DIR / "official_page_snapshot_log.csv", snapshot_rows)

supplemental_rows = [{
    "country_iso3":"EGY", "year":row["year"], "indicator_code":"ST.INT.ARVL",
    "indicator_name":"International tourism, number of arrivals", "value":row["value"], "unit":"persons",
    "source_name":row["source_name"], "source_url":row["source_url"], "license_terms":"Public government page",
    "source_method":"OFFICIAL_WEB_PUBLICATION_NORMALIZED", "precision_flag":row["precision_flag"],
    "evidence_note":row["evidence_note"], "downloaded_utc":NOW_UTC,
    "qa_status":"READY_OFFICIAL_ANNUAL_WITH_PRECISION_FLAG",
} for row in OFFICIAL_ARRIVALS]
write_csv(NORMALIZED_DIR / "fact_official_arrivals_2020_2025.csv", supplemental_rows)

# Government rows take precedence over overlapping WDI arrival years.
supplement_keys = {(r["indicator_code"], r["year"]) for r in supplemental_rows}
tourism_annual = [r for r in wb_rows if (r["indicator_code"], r["year"]) not in supplement_keys] + supplemental_rows
tourism_annual.sort(key=lambda r: (r["indicator_code"], r["year"]))
write_csv(NORMALIZED_DIR / "fact_tourism_annual_verified_long.csv", tourism_annual)
event("SRC_EGY_OFFICIAL_ANNUAL", "SUCCESS", len(supplemental_rows), "Official values normalized; precision retained.")
print(f"Combined verified annual rows: {len(tourism_annual)}")

## 2) أدلة وزارة السياحة والآثار MOTA

الـendpoint العام يعيد صفحات JSON. يتم حفظ كل صفحة، إزالة تكرار رابط السجل الرسمي، وإنشاء ID ثابت من الرابط. الصفوف التي لا يمكن ربطها بمحافظة مصرية صحيحة تُحفظ في `exclusions` بدل إدخالها في جدول التحليل.

In [ ]:
MOTA_ENDPOINT = "https://mota.gov.eg/GenericList/GetAll"
MOTA_CONFIG = [
    {"key":"hotels","page_id":"27953","prefix":"MOTA-FH-","id_col":"mota_property_id","name_ar_col":"property_name_ar","name_en_col":"property_name_en","output":"dim_mota_fixed_accommodation_official","page_url":"https://mota.gov.eg/ar/الشركات-والمنشآت-الفندقية-والسياحية/دليل-المنشآت-الفندقية-والمحال-والأنشطة-السياحية/دليل-المنشآت-الفندقية-الثابتة-والمنتجعات-السياحية/"},
    {"key":"archaeological_sites","page_id":"17739","prefix":"MOTA-AS-","id_col":"mota_site_id","name_ar_col":"site_name_ar","name_en_col":"site_name_en","output":"dim_mota_open_archaeological_sites_official","page_url":"https://mota.gov.eg/ar/الآثار-والمتاحف/المجلس-الأعلى-للآثار/دليل-المواقع-الأثرية-المفتوحة-للزيارة-جديد/"},
    {"key":"museums","page_id":"17737","prefix":"MOTA-MU-","id_col":"mota_museum_id","name_ar_col":"museum_name_ar","name_en_col":"museum_name_en","output":"dim_mota_open_museums_official","page_url":"https://mota.gov.eg/ar/الآثار-والمتاحف/المجلس-الأعلى-للآثار/دليل-متاحف-الآثار-المفتوحة-للزيارة-جديد/"},
]

AR_GOV_TO_EN = {
    "القاهرة":"Cairo","الجيزة":"Giza","القليوبية":"Qalyubia","الإسكندرية":"Alexandria","الاسكندرية":"Alexandria","الأسكندرية":"Alexandria",
    "البحيرة":"Beheira","كفر الشيخ":"Kafr El Sheikh","الغربية":"Gharbia","المنوفية":"Monufia","الدقهلية":"Dakahlia","دمياط":"Damietta",
    "الشرقية":"Sharqia","بورسعيد":"Port Said","بور سعيد":"Port Said","الإسماعيلية":"Ismailia","الاسماعيلية":"Ismailia","السويس":"Suez",
    "بني سويف":"Beni Suef","الفيوم":"Fayoum","المنيا":"Minya","أسيوط":"Assiut","اسيوط":"Assiut","سوهاج":"Sohag","قنا":"Qena",
    "الأقصر":"Luxor","الاقصر":"Luxor","أسوان":"Aswan","اسوان":"Aswan","البحر الأحمر":"Red Sea","البحر الاحمر":"Red Sea",
    "جنوب سيناء":"South Sinai","شمال سيناء":"North Sinai","مطروح":"Matrouh","الوادي الجديد":"New Valley",
}

def mota_page(page_id, page_number=None):
    body = {"genericListId":page_id,"Lang":"ar-EG","TextSearch":"","GovernateId":0,"CityId":0,"CategoryId":0,"DistrictId":0,"Rating":0,"DateFrom":"","DateTo":""}
    if page_number is not None:
        body["PageNumber"] = page_number
    response, _, _ = http_json(MOTA_ENDPOINT, method="POST", json_body=body, headers={"Referer":"https://mota.gov.eg/","Accept":"application/json, text/plain, */*"}, timeout=90)
    if not response.get("success") or not response.get("data"):
        raise ValueError(response.get("errorMessage") or "MOTA returned no data")
    return response

def collect_mota_directory(config):
    first = mota_page(config["page_id"])
    group = first["data"][0]
    total = int(group.get("totalCount") or 0)
    first_items = list(group.get("genericListItems") or [])
    if not first_items:
        raise ValueError(f"MOTA {config['key']} returned zero rows")
    page_count = math.ceil(total / len(first_items))
    pages = {1:first}
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(mota_page, config["page_id"], page):page for page in range(2, page_count + 1)}
        for future in as_completed(futures):
            pages[futures[future]] = future.result()
    all_items = []
    page_folder = RAW_DIR / "mota" / config["key"]
    for page_number in sorted(pages):
        write_json(page_folder / f"page_{page_number:03d}.json", pages[page_number])
        for item in pages[page_number]["data"][0].get("genericListItems") or []:
            item = dict(item)
            item["source_page_number"] = page_number
            all_items.append(item)
    write_json(RAW_DIR / "mota" / f"{config['key']}_raw_all.json", {"source_endpoint":MOTA_ENDPOINT,"page_id":config["page_id"],"reported_total_count":total,"downloaded_utc":NOW_UTC,"items":all_items})

    unique = {}
    duplicates = []
    for item in all_items:
        url = clean_text(item.get("itemUrl"))
        if not url:
            url = f"missing-url:{config['key']}:{len(unique)}:{clean_text(item.get('pageTitle'))}"
        if url in unique:
            duplicates.append(item)
        else:
            unique[url] = item
    normalized = []
    for url, item in unique.items():
        gov_ar = clean_text(item.get("governate"))
        gov_en = AR_GOV_TO_EN.get(gov_ar, "")
        rating_raw = item.get("rating", "")
        try:
            stars = int(float(rating_raw))
        except (TypeError, ValueError):
            stars = ""
        if stars not in (1,2,3,4,5):
            stars = ""
        record = {
            config["id_col"]:stable_id(config["prefix"], url),
            config["name_ar_col"]:clean_text(item.get("pageTitle")),
            config["name_en_col"]:clean_text(item.get("briefDescription")),
            "governorate_en":gov_en,"governorate_ar":gov_ar,"city_ar":clean_text(item.get("city")),
            "official_rating_raw":rating_raw,"official_classification_stars":stars,
            "license_number":clean_text(item.get("licenseNumber")),"official_record_url":clean_text(item.get("itemUrl")),
            "source_page_number":item.get("source_page_number"),"source_name":"Egypt Ministry of Tourism and Antiquities public directory",
            "source_url":config["page_url"],"source_method":"AUTOMATED_OFFICIAL_PUBLIC_ENDPOINT","downloaded_utc":NOW_UTC,
            "geography_qa_status":"READY_GOVERNORATE_MAPPED" if gov_en else "REVIEW_GOVERNORATE_UNMAPPED",
            "qa_status":"READY_OFFICIAL_IDENTITY_RECORD" if gov_en else "EXCLUDED_FROM_ADM1_ANALYSIS",
        }
        normalized.append(record)
    ready = [r for r in normalized if r["geography_qa_status"] == "READY_GOVERNORATE_MAPPED"]
    excluded = [r for r in normalized if r["geography_qa_status"] != "READY_GOVERNORATE_MAPPED"]
    write_csv(NORMALIZED_DIR / f"{config['output']}_full.csv", normalized)
    write_csv(NORMALIZED_DIR / f"{config['output']}.csv", ready)
    write_csv(EXCLUSIONS_DIR / f"{config['key']}_unmapped_governorate.csv", excluded)
    write_csv(EXCLUSIONS_DIR / f"{config['key']}_duplicate_page_rows.csv", duplicates)
    event("SRC_MOTA_DIRECTORIES", "SUCCESS", len(ready), f"{config['key']}: reported={total}; unique={len(normalized)}; excluded={len(excluded)}")
    return {"config":config,"reported":total,"unique":len(normalized),"ready":ready,"excluded":excluded}

mota_results = [collect_mota_directory(config) for config in MOTA_CONFIG] if RUN_MOTA else []
for result in mota_results:
    print(result["config"]["key"], {"reported":result["reported"],"unique":result["unique"],"ready":len(result["ready"]),"excluded":len(result["excluded"])})

In [ ]:
# Official accommodation inventory aggregation: counts only, never rooms/occupancy/demand.
hotel_result = next((r for r in mota_results if r["config"]["key"] == "hotels"), None)
inventory_rows = []
if hotel_result:
    counts = Counter((r["governorate_en"], r["governorate_ar"], str(r["official_classification_stars"] or "UNCLASSIFIED")) for r in hotel_result["ready"])
    for (gov_en, gov_ar, classification), count in sorted(counts.items()):
        inventory_rows.append({
            "governorate_en":gov_en,"governorate_ar":gov_ar,"official_classification":classification,
            "official_directory_properties":count,
            "metric_definition":"Count of unique fixed accommodation records in the MOTA public directory snapshot",
            "source_url":hotel_result["config"]["page_url"],"snapshot_utc":NOW_UTC,
            "qa_status":"READY_OFFICIAL_DIRECTORY_SNAPSHOT",
            "allowed_use":"Inventory counts by governorate/classification; not rooms, beds, occupancy, guests or demand.",
        })
write_csv(NORMALIZED_DIR / "fact_mota_fixed_accommodation_inventory_by_governorate.csv", inventory_rows)
print(f"MOTA inventory groups: {len(inventory_rows)}")

## 3) حدود المحافظات من geoBoundaries

يتم أولًا تنزيل metadata الرسمي ثم رابط GeoJSON الموجود داخله، بدل تثبيت رابط ملف قد يتغير.

In [ ]:
def collect_geoboundaries():
    api_url = "https://www.geoboundaries.org/api/current/gbOpen/EGY/ADM1/"
    metadata, _, _ = http_json(api_url, timeout=90)
    if isinstance(metadata, list):
        metadata = metadata[0]
    write_json(RAW_DIR / "geoboundaries" / "EGY_ADM1_metadata.json", metadata)
    geojson_url = metadata.get("gjDownloadURL") or metadata.get("simplifiedGeometryGeoJSON")
    if not geojson_url:
        raise ValueError("geoBoundaries metadata has no GeoJSON download URL")
    payload, final_url, _ = http_bytes(geojson_url, timeout=120)
    geojson = json.loads(payload.decode("utf-8-sig"))
    write_json(NORMALIZED_DIR / "EGY_ADM1.geojson", geojson)
    feature_count = len(geojson.get("features") or [])
    if feature_count < 20:
        raise ValueError(f"Unexpected ADM1 feature count: {feature_count}")
    event("SRC_GEOBOUNDARIES", "SUCCESS", feature_count, final_url)
    return feature_count

geoboundaries_features = collect_geoboundaries() if RUN_GEOBOUNDARIES else 0
print(f"geoBoundaries ADM1 features: {geoboundaries_features}")

## 4) مناخ الوجهات من NASA POWER

تُستخدم إحداثيات `Data/dim_destination_reference_adm1.csv`. يُستبعد المفتاح الشهري رقم 13 لأنه ملخص سنوي، ويُشترط وجود 12 شهرًا لكل سنة ولكل متغير.

In [ ]:
NASA_ENDPOINT = "https://power.larc.nasa.gov/api/temporal/monthly/point"
NASA_PARAMETERS = ["T2M","T2M_MAX","T2M_MIN","PRECTOTCORR","WS2M","RH2M"]

def nasa_url(destination):
    return NASA_ENDPOINT + "?" + urlencode({
        "parameters":",".join(NASA_PARAMETERS),"community":"RE",
        "longitude":destination["longitude"],"latitude":destination["latitude"],
        "start":START_YEAR,"end":END_YEAR,"format":"JSON",
    })

def fetch_nasa(destination):
    url = nasa_url(destination)
    payload, final_url, _ = http_json(url, timeout=150)
    if "properties" not in payload or "parameter" not in payload["properties"]:
        raise ValueError(f"Unexpected NASA response: {destination['destination_id']}")
    return destination, final_url, payload

def collect_nasa_power():
    destinations = read_csv(DATA_DIR / "dim_destination_reference_adm1.csv")
    completed = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = [pool.submit(fetch_nasa, row) for row in destinations]
        for future in as_completed(futures):
            completed.append(future.result())
    expected = {f"{year}{month:02d}" for year in range(START_YEAR, END_YEAR + 1) for month in range(1,13)}
    rows = []
    for destination, url, payload in sorted(completed, key=lambda x: x[0]["destination_id"]):
        write_json(RAW_DIR / "nasa_power" / f"{safe_filename(destination['destination_id'])}.json", payload)
        parameter_data = payload["properties"]["parameter"]
        missing = set(NASA_PARAMETERS) - set(parameter_data)
        if missing:
            raise ValueError(f"NASA missing {sorted(missing)} for {destination['destination_id']}")
        for parameter in NASA_PARAMETERS:
            actual = {key for key in parameter_data[parameter] if len(key) == 6 and key[-2:] in {f"{m:02d}" for m in range(1,13)}}
            if actual != expected:
                raise ValueError(f"Incomplete NASA monthly series for {destination['destination_id']} / {parameter}")
        units = payload.get("parameters", {})
        for key in sorted(expected):
            row = {
                "destination_id":destination["destination_id"],"destination_en":destination["destination_en"],"destination_ar":destination.get("destination_ar", ""),
                "governorate_en":destination["governorate_en"],"governorate_ar":destination.get("governorate_ar", ""),
                "latitude":destination["latitude"],"longitude":destination["longitude"],"year":int(key[:4]),"month":int(key[4:]),
            }
            for parameter in NASA_PARAMETERS:
                value = parameter_data[parameter][key]
                row[parameter.lower()] = "" if value == -999 else value
                row[f"{parameter.lower()}_unit"] = units.get(parameter, {}).get("units", "API_REPORTED")
            row.update({"data_source":"NASA POWER Monthly Point API","source_url":url,"source_basis":"MERRA-2 modelled climate; not local station observation","source_method":"AUTOMATED_OFFICIAL_API","downloaded_utc":NOW_UTC,"qa_status":"READY_OFFICIAL_MODELLED_CLIMATE"})
            rows.append(row)
    expected_count = len(destinations) * (END_YEAR - START_YEAR + 1) * 12
    if len(rows) != expected_count:
        raise ValueError(f"NASA row count {len(rows)} != expected {expected_count}")
    write_csv(NORMALIZED_DIR / "fact_destination_climate_monthly_nasa_power.csv", rows)
    event("SRC_NASA_POWER", "SUCCESS", len(rows), f"{len(destinations)} destinations x 84 months")
    return rows

nasa_rows = collect_nasa_power() if RUN_NASA_POWER else []
print(f"NASA POWER monthly rows: {len(nasa_rows)}")

## 5) اهتمام Wikimedia الشهري

العناوين تأتي من جدول التغطية المعتمد في `Data`. الطلب يستبعد bots وspiders باستخدام agent=`user`. يتم تكوين جدول تغطية جديد ونافذة مقارنة مشتركة تبدأ من يوليو 2024 وتنتهي عند آخر شهر مكتمل.

In [ ]:
WIKIMEDIA_ENDPOINT = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"

def wikimedia_url(project, title):
    encoded = quote(title.replace(" ", "_"), safe="")
    return f"{WIKIMEDIA_ENDPOINT}/{project}/all-access/user/{encoded}/monthly/{WIKIMEDIA_START}/{WIKIMEDIA_END}"

def fetch_wikimedia(row):
    title = row["wikipedia_title"]
    project = row.get("wiki_project") or "en.wikipedia"
    url = wikimedia_url(project, title)
    payload, final_url, _ = http_json(url, timeout=90, attempts=7)
    return title, project, final_url, payload

def month_range(start, end):
    start_y, start_m = map(int, start[:7].split("-"))
    end_y, end_m = map(int, end[:7].split("-"))
    result = []
    y, m = start_y, start_m
    while (y, m) <= (end_y, end_m):
        result.append(f"{y:04d}-{m:02d}-01")
        m += 1
        if m == 13:
            y, m = y + 1, 1
    return result

def collect_wikimedia():
    titles = read_csv(DATA_DIR / "dim_wikimedia_title_coverage.csv")
    # Explicitly enforce the current comparable scope.
    titles = [r for r in titles if (r.get("wiki_project") or "en.wikipedia") == "en.wikipedia"]
    completed = []
    # One worker avoids bursts and makes the public endpoint reproducible.
    with ThreadPoolExecutor(max_workers=1) as pool:
        futures = [pool.submit(fetch_wikimedia, row) for row in titles]
        for future in as_completed(futures):
            completed.append(future.result())
    rows = []
    coverage = []
    for title, project, url, payload in sorted(completed):
        write_json(RAW_DIR / "wikimedia" / f"{safe_filename(title)}.json", payload)
        title_rows = []
        for item in payload.get("items") or []:
            timestamp = str(item["timestamp"])
            month_date = f"{timestamp[:4]}-{timestamp[4:6]}-01"
            if month_date > COMMON_WINDOW_END:
                continue
            record = {
                "wikipedia_title":title,"wiki_project":project,"date":month_date,"year":int(timestamp[:4]),"month_num":int(timestamp[4:6]),
                "views":int(item["views"]),"metric_name":"Wikimedia pageviews","metric_scope":"all-access; user agent; monthly",
                "source_url":url,"downloaded_utc":NOW_UTC,"qa_status":"READY_DIGITAL_INTEREST_NOT_VISITS",
            }
            title_rows.append(record)
            rows.append(record)
        dates = sorted(r["date"] for r in title_rows)
        if dates:
            expected = month_range(dates[0], dates[-1])
            missing = sorted(set(expected) - set(dates))
            common_expected = month_range(COMMON_WINDOW_START, COMMON_WINDOW_END)
            common_eligible = all(d in set(dates) for d in common_expected)
            coverage.append({
                "wikipedia_title":title,"wiki_project":project,"coverage_start":dates[0],"coverage_end":dates[-1],"available_months":len(dates),
                "expected_months_from_start":len(expected),"internal_missing_months":len(missing),"coverage_group":"FULL_FROM_2019_01" if dates[0] == "2019-01-01" and not missing else "PARTIAL_OR_GAPPED",
                "common_window_start":COMMON_WINDOW_START,"common_window_end":COMMON_WINDOW_END,"common_window_eligible":"ELIGIBLE" if common_eligible else "NOT_ELIGIBLE",
                "comparison_use":"Use for digital-interest trend only.","source_url":"https://wikimedia.org/api/rest_v1/","qa_status":"READY_COVERAGE_DOCUMENTED","generated_utc":NOW_UTC,
            })
    common_titles = {r["wikipedia_title"] for r in coverage if r["common_window_eligible"] == "ELIGIBLE"}
    common_rows = [r for r in rows if r["wikipedia_title"] in common_titles and COMMON_WINDOW_START <= r["date"] <= COMMON_WINDOW_END]
    write_csv(NORMALIZED_DIR / "fact_wikimedia_reference_interest_monthly.csv", rows)
    write_csv(NORMALIZED_DIR / "dim_wikimedia_title_coverage.csv", coverage)
    write_csv(NORMALIZED_DIR / "fact_wikimedia_interest_common_window.csv", common_rows)
    event("SRC_WIKIMEDIA", "SUCCESS", len(rows), f"{len(coverage)} English titles; {len(common_titles)} common-window titles")
    return rows, coverage, common_rows

wikimedia_rows, wikimedia_coverage, wikimedia_common = collect_wikimedia() if RUN_WIKIMEDIA else ([],[],[])
print(f"Wikimedia rows: {len(wikimedia_rows)}; titles: {len(wikimedia_coverage)}; common-window rows: {len(wikimedia_common)}")

## 6) مواقع التراث العالمي من UNESCO

يتم تنزيل XML الرسمي الكامل ثم تصفية السجلات التي يحمل `iso_code` فيها الرمز `eg`. تتم إضافة المحافظة من طبقة المرجع المعتمدة الموجودة في `Data` عند تطابق اسم الموقع، لأن XML الرسمي لا يقدم ADM1 جاهزًا.

In [ ]:
def xml_text(row, tag):
    node = row.find(tag)
    return clean_text(node.text if node is not None else "")

def collect_unesco():
    url = "https://whc.unesco.org/en/list/xml/"
    payload, final_url, _ = http_bytes(url, timeout=180)
    xml_path = RAW_DIR / "unesco" / "world_heritage_list.xml"
    xml_path.parent.mkdir(parents=True, exist_ok=True)
    xml_path.write_bytes(payload)
    root = ET.fromstring(payload)
    gov_lookup = {}
    approved_path = DATA_DIR / "dim_unesco_world_heritage_sites.csv"
    if approved_path.exists():
        gov_lookup = {clean_text(r.get("whs_name")):r.get("governorate_en", "") for r in read_csv(approved_path)}
    rows = []
    for node in root.findall(".//row"):
        iso_code = xml_text(node, "iso_code").lower()
        states = xml_text(node, "states").lower()
        iso_parts = {part.strip() for part in re.split(r"[,; ]+", iso_code) if part.strip()}
        if "eg" not in iso_parts and "egypt" not in states:
            continue
        name = xml_text(node, "site")
        site_id = xml_text(node, "id_number")
        rows.append({
            "unesco_id":site_id,"whs_name":name,"year_inscribed":xml_text(node, "date_inscribed"),"category":xml_text(node, "category").lower(),
            "governorate_en":gov_lookup.get(name, ""),"latitude":xml_text(node, "latitude"),"longitude":xml_text(node, "longitude"),
            "criteria":xml_text(node, "criteria_txt"),"danger":xml_text(node, "danger"),"area_hectares":xml_text(node, "area_hectares"),
            "official_record_url":f"https://whc.unesco.org/en/list/{site_id}","source_url":url,"source_method":"AUTOMATED_OFFICIAL_XML",
            "downloaded_utc":NOW_UTC,"qa_status":"READY_OFFICIAL_REFERENCE",
        })
    if len(rows) < 7:
        raise ValueError(f"Expected at least 7 UNESCO Egypt records, received {len(rows)}")
    write_csv(NORMALIZED_DIR / "dim_unesco_world_heritage_sites.csv", rows)
    event("SRC_UNESCO", "SUCCESS", len(rows), final_url)
    return rows

unesco_rows = collect_unesco() if RUN_UNESCO else []
print(f"UNESCO Egypt records: {len(unesco_rows)}")

## 7) OpenStreetMap عبر Overpass — اختياري

الكود التالي كامل ويجمع مرشحي المعالم والإقامة داخل مصر. هو **متوقف افتراضيًا** لأن Overpass خدمة عامة لها حدود تحميل وقد تتوقف مؤقتًا. لتشغيله غيّر `RUN_OSM = True` في خلية الإعدادات ثم شغّل هذه الخلية. المخرجات مرجعية فقط ويجب ألا تُعامل كسجل حكومي أو مقياس طلب.

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSM_QUERIES = {
    "attractions": """[out:json][timeout:300];area[\"ISO3166-1\"=\"EG\"][admin_level=2]->.egypt;(nwr[\"tourism\"~\"^(attraction|museum|gallery|zoo|aquarium|theme_park)$\"](area.egypt);nwr[\"historic\"~\"^(archaeological_site|monument|castle|ruins|tomb|memorial)$\"](area.egypt););out center tags;""",
    "accommodation": """[out:json][timeout:300];area[\"ISO3166-1\"=\"EG\"][admin_level=2]->.egypt;nwr[\"tourism\"~\"^(hotel|hostel|guest_house|motel|resort|apartment|camp_site|chalet)$\"](area.egypt);out center tags;""",
}

def collect_overpass(label, query):
    url = OVERPASS_URL + "?" + urlencode({"data":query})
    payload, final_url, _ = http_json(url, timeout=360, attempts=5)
    write_json(RAW_DIR / "osm" / f"egypt_{label}_overpass.json", payload)
    rows = []
    for element in payload.get("elements") or []:
        tags = element.get("tags") or {}
        center = element.get("center") or {}
        latitude = element.get("lat", center.get("lat", ""))
        longitude = element.get("lon", center.get("lon", ""))
        row = {
            "osm_type":element.get("type"),"osm_id":element.get("id"),"latitude":latitude,"longitude":longitude,
            "name":tags.get("name", ""),"name_en":tags.get("name:en", ""),"name_ar":tags.get("name:ar", ""),
            "tourism":tags.get("tourism", ""),"historic":tags.get("historic", ""),"amenity":tags.get("amenity", ""),
            "wikidata":tags.get("wikidata", ""),"wikipedia":tags.get("wikipedia", ""),"website":tags.get("website", tags.get("contact:website", "")),
            "phone":tags.get("phone", tags.get("contact:phone", "")),"stars":tags.get("stars", ""),"rooms":tags.get("rooms", ""),
            "addr_city":tags.get("addr:city", ""),"addr_governorate":tags.get("addr:governorate", ""),
            "osm_url":f"https://www.openstreetmap.org/{element.get('type')}/{element.get('id')}",
            "source_method":"AUTOMATED_OVERPASS_QUERY","downloaded_utc":NOW_UTC,"qa_status":"REFERENCE_ONLY_REQUIRES_IDENTITY_AND_GEOGRAPHY_QA",
        }
        rows.append(row)
    write_csv(NORMALIZED_DIR / f"osm_egypt_{label}_reference.csv", rows)
    event("SRC_OSM", "SUCCESS", len(rows), f"{label}; ODbL; reference only")
    return rows

osm_outputs = {}
if RUN_OSM:
    for label, query in OSM_QUERIES.items():
        osm_outputs[label] = collect_overpass(label, query)
        print(f"OSM {label}: {len(osm_outputs[label])}")
else:
    event("SRC_OSM", "SKIPPED_BY_DEFAULT", 0, "Set RUN_OSM=True to run the complete Overpass queries; public service is rate-limited.")
    print("OSM skipped by default. Set RUN_OSM=True when a fresh open-reference snapshot is needed.")

## 8) اختبارات الجودة وManifest

الاختبارات التالية لا تثبت أن كل قيمة في العالم صحيحة، لكنها تمنع الأخطاء الشائعة: ملف فارغ، عدد شهور ناقص، تكرار مفتاح، مصدر غير موثق، أو تغير كبير غير متوقع في عدد السجلات.

In [ ]:
QA_CHECKS = []
def qa(check_id, passed, observed, expected, severity="ERROR"):
    QA_CHECKS.append({"check_id":check_id,"status":"PASS" if passed else "FAIL","observed":observed,"expected":expected,"severity":severity,"checked_utc":NOW_UTC})

if RUN_WORLD_BANK:
    qa("WB_NONEMPTY", len(wb_rows) > 40, len(wb_rows), ">40")
    qa("TOURISM_ARRIVALS_2020_2025", {r["year"] for r in supplemental_rows} == set(range(2020,2026)), sorted(r["year"] for r in supplemental_rows), "2020..2025")
    keys = [(r["indicator_code"], r["year"]) for r in tourism_annual]
    qa("TOURISM_UNIQUE_INDICATOR_YEAR", len(keys) == len(set(keys)), len(keys)-len(set(keys)), "0 duplicates")
if RUN_MOTA:
    by_key = {r["config"]["key"]:r for r in mota_results}
    qa("MOTA_HOTELS_MINIMUM", len(by_key["hotels"]["ready"]) >= 900, len(by_key["hotels"]["ready"]), ">=900")
    qa("MOTA_ARCHAEOLOGY_MINIMUM", len(by_key["archaeological_sites"]["ready"]) >= 150, len(by_key["archaeological_sites"]["ready"]), ">=150")
    qa("MOTA_MUSEUMS_MINIMUM", len(by_key["museums"]["ready"]) >= 30, len(by_key["museums"]["ready"]), ">=30")
if RUN_GEOBOUNDARIES:
    qa("GEOBOUNDARIES_FEATURES", geoboundaries_features >= 20, geoboundaries_features, ">=20 ADM1 features")
if RUN_NASA_POWER:
    destination_count = len(read_csv(DATA_DIR / "dim_destination_reference_adm1.csv"))
    expected_nasa = destination_count * (END_YEAR - START_YEAR + 1) * 12
    qa("NASA_COMPLETE_MONTHS", len(nasa_rows) == expected_nasa, len(nasa_rows), expected_nasa)
if RUN_WIKIMEDIA:
    common_expected_months = len(month_range(COMMON_WINDOW_START, COMMON_WINDOW_END))
    eligible_titles = sum(1 for r in wikimedia_coverage if r["common_window_eligible"] == "ELIGIBLE")
    qa("WIKIMEDIA_COMMON_WINDOW", len(wikimedia_common) == eligible_titles * common_expected_months, len(wikimedia_common), eligible_titles * common_expected_months)
    qa("WIKIMEDIA_ENGLISH_ONLY", all(r["wiki_project"] == "en.wikipedia" for r in wikimedia_rows), sorted(set(r["wiki_project"] for r in wikimedia_rows)), "['en.wikipedia']")
if RUN_UNESCO:
    qa("UNESCO_EGYPT_MINIMUM", len(unesco_rows) >= 7, len(unesco_rows), ">=7")

write_csv(OUTPUT_DIR / "qa_checks.csv", QA_CHECKS)
write_csv(OUTPUT_DIR / "collection_log.csv", COLLECTION_EVENTS)

manifest = []
for path in sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file() and p.name != "file_manifest.csv"):
    row_count = ""
    if path.suffix.lower() == ".csv":
        try:
            with path.open("r", encoding="utf-8-sig", newline="") as handle:
                row_count = max(0, sum(1 for _ in handle) - 1)
        except Exception:
            row_count = "UNREADABLE"
    manifest.append({
        "relative_path":str(path.relative_to(OUTPUT_DIR)).replace("\\", "/"),"bytes":path.stat().st_size,
        "sha256":sha256_file(path),"csv_data_rows":row_count,"generated_utc":NOW_UTC,
    })
write_csv(OUTPUT_DIR / "file_manifest.csv", manifest)

failed = [r for r in QA_CHECKS if r["status"] == "FAIL" and r["severity"] == "ERROR"]
print(f"QA checks: {len(QA_CHECKS)}; failed errors: {len(failed)}; files in manifest: {len(manifest)}")
if failed:
    raise AssertionError("QA failed: " + json.dumps(failed, ensure_ascii=False))
print("COLLECTION COMPLETE — all enabled sections passed QA.")

## تفسير المخرجات

- استخدم `output/normalized` لمراجعة نتيجة جمع جديدة.
- لا تستبدل ملفات `Data` تلقائيًا. قارِن الأعداد والبصمات واختبارات الجودة أولًا، ثم انقل نسخة جديدة إلى `Data` بعد الاعتماد.
- احتفظ بـ`output/raw` لإثبات مصدر كل سجل وإعادة فحص التحويل.
- احتفظ بـ`output/exclusions` لإظهار ما تم استبعاده ولماذا؛ لا تحذف الصفوف غير الواقعية بصمت.
- عند مناقشة النتائج، سمِّ مقياس Wikimedia «اهتمام رقمي»، وسمِّ NASA «مناخ نموذجي»، وسمِّ MOTA «سجل/دليل رسمي»، ولا تحول أيًا منها إلى عدد زوار أو إشغال دون مصدر مناسب.